# Treinamento de Modelo de Classificação de Tosse - ICBHI Dataset

Este notebook processa o dataset ICBHI diretamente no Google Colab, usando API Kaggle para download automático, e treina um modelo MobileNetV2 para classificação de Pneumonia e Bronquite.

## Fluxo de Trabalho:
1. Setup inicial e instalação de dependências
2. Configuração da API Kaggle e download do dataset
3. Processamento de dados (filtragem, padronização, extração de características)
4. Treinamento do modelo MobileNetV2
5. Conversão para TensorFlow Lite
6. Download do modelo final


In [ ]:
# Célula 1: Setup Inicial e Instalação de Dependências

# Instala dependências necessárias
%pip install -q kaggle librosa scipy scikit-learn tqdm

# Importa bibliotecas
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import zipfile

# Configura GPU
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

# Configura paths
PROJECT_ROOT = Path('/content/tosse')
PROJECT_ROOT.mkdir(exist_ok=True)

# Adiciona ao path do Python
sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Setup inicial concluído!")


## 2. Configuração da API Kaggle e Download do Dataset


In [ ]:
# Célula 2: Configuração Kaggle e Download do Dataset ICBHI

# Opção 1: Upload do arquivo kaggle.json
# Descomente e execute apenas uma vez para fazer upload do arquivo
# from google.colab import files
# uploaded = files.upload()
# for fn in uploaded.keys():
#     if fn == 'kaggle.json':
#         os.makedirs('/root/.kaggle', exist_ok=True)
#         os.rename(fn, '/root/.kaggle/kaggle.json')
#         os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Opção 2: Configurar credenciais manualmente
# Descomente e preencha com suas credenciais
# import json
# kaggle_credentials = {
#     "username": "seu_usuario_kaggle",
#     "key": "sua_chave_api_kaggle"
# }
# os.makedirs('/root/.kaggle', exist_ok=True)
# with open('/root/.kaggle/kaggle.json', 'w') as f:
#     json.dump(kaggle_credentials, f)
# os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Importa módulo de setup Kaggle
from src.utils.kaggle_setup import setup_kaggle_credentials, download_icbhi_dataset, find_diagnosis_file

# Configura credenciais (ajuste conforme necessário)
# setup_kaggle_credentials(username="seu_usuario", key="sua_chave")

# Baixa dataset ICBHI
DATASET_NAME = "vbookshelf/icbhi-2017-respiratory-sound-database"
DATA_DIR = download_icbhi_dataset(
    dataset_name=DATASET_NAME,
    output_dir="/content/tmp/icbhi",
    unzip=True
)

# Encontra arquivo de diagnósticos
diagnosis_file = find_diagnosis_file(DATA_DIR)
if diagnosis_file:
    print(f"✅ Arquivo de diagnósticos encontrado: {diagnosis_file}")
else:
    print("⚠️ Arquivo de diagnósticos não encontrado. Verifique a estrutura do dataset.")

print(f"\n✅ Dataset baixado em: {DATA_DIR}")


## 3. Processamento de Dados


In [ ]:
# Célula 3: Processamento de Dados

# Copia código fonte para o Colab (ou faz upload do projeto)
# Para este exemplo, vamos executar o script diretamente

from src.data.process_icbhi_dataset import process_icbhi_dataset

# Processa dataset completo
OUTPUT_DIR = Path("/content/processed_data")

process_icbhi_dataset(
    data_dir=str(DATA_DIR),
    diagnosis_csv=str(diagnosis_file),
    output_dir=str(OUTPUT_DIR),
    sample_rate=16000,
    use_butterworth=True,
    butterworth_cutoff=8000.0,
    extract_mfcc=True,
    extract_mel=True,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15
)

print(f"\n✅ Dados processados salvos em: {OUTPUT_DIR}")


## 4. Carregamento de Dados Processados e Visualização


In [ ]:
# Célula 4: Carregamento e Visualização

from src.training.train import load_processed_data
import matplotlib.pyplot as plt

# Carrega dados processados
(X_train, X_val, X_test, y_train, y_val, y_test), label_encoder = load_processed_data(
    str(OUTPUT_DIR),
    feature_type='mel'
)

print(f"Shape dos dados de treino: {X_train.shape}")
print(f"Shape dos dados de validação: {X_val.shape}")
print(f"Shape dos dados de teste: {X_test.shape}")
print(f"Classes: {label_encoder.classes_}")

# Visualiza alguns espectrogramas
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i in range(4):
    ax = axes[i // 2, i % 2]
    sample_idx = np.random.randint(0, len(X_train))
    spectrogram = X_train[sample_idx, :, :, 0]
    im = ax.imshow(spectrogram, aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(f'Amostra {sample_idx} - Classe: {label_encoder.classes_[np.argmax(y_train[sample_idx])]}')
    ax.set_xlabel('Tempo')
    ax.set_ylabel('Frequência Mel')
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("✅ Dados carregados e visualizados!")


## 5. Treinamento do Modelo MobileNetV2


In [ ]:
# Célula 5: Treinamento do Modelo

from src.models.cough_classifier import CoughClassifier
from tensorflow import keras
from pathlib import Path

# Determina input shape
input_shape = X_train.shape[1:]
num_classes = len(label_encoder.classes_)

print(f"Input shape: {input_shape}")
print(f"Número de classes: {num_classes}")

# Cria modelo MobileNetV2
classifier = CoughClassifier(
    input_shape=input_shape,
    num_classes=num_classes,
    model_type='mobilenet'  # Usa MobileNetV2 otimizado para mobile
)

model = classifier.get_model()
model.summary()

# Configura callbacks
MODEL_DIR = Path("/content/models")
MODEL_DIR.mkdir(exist_ok=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(MODEL_DIR / 'best_model.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

# Treina modelo
EPOCHS = 50
BATCH_SIZE = 32

print("\n🚀 Iniciando treinamento...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# Avalia no conjunto de teste
print("\n📊 Avaliando no conjunto de teste...")
test_loss, test_acc, _ = model.evaluate(X_test, y_test, verbose=1)
print(f"Teste - Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}")

# Plota histórico de treinamento
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Treino')
plt.plot(history.history['val_accuracy'], label='Validação')
plt.title('Acurácia do Modelo')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.title('Loss do Modelo')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

print("✅ Treinamento concluído!")


In [ ]:
# Célula 6: Conversão para TensorFlow Lite

# Converte modelo para TFLite
TFLITE_DIR = MODEL_DIR / 'tflite'
TFLITE_DIR.mkdir(exist_ok=True)

print("🔄 Convertendo modelo para TensorFlow Lite...")

# Quantização float16 (recomendado para mobile)
classifier.convert_to_tflite(
    output_path=str(TFLITE_DIR / 'cough_classifier_fp16.tflite'),
    quantization='float16'
)

# Quantização int8 (menor tamanho, pode precisar de dataset representativo)
# classifier.convert_to_tflite(
#     output_path=str(TFLITE_DIR / 'cough_classifier_int8.tflite'),
#     quantization='int8'
# )

# Verifica tamanho dos arquivos
import os
for file in TFLITE_DIR.glob('*.tflite'):
    size_mb = os.path.getsize(file) / (1024 * 1024)
    print(f"  ✅ {file.name}: {size_mb:.2f} MB")

print("✅ Conversão concluída!")


## 7. Download e Persistência


In [ ]:
# Célula 7: Download e Persistência

# Opção 1: Salvar no Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/tosse_models')
    DRIVE_MODEL_DIR.mkdir(exist_ok=True)
    
    # Copia modelo TFLite
    import shutil
    for tflite_file in TFLITE_DIR.glob('*.tflite'):
        shutil.copy(tflite_file, DRIVE_MODEL_DIR / tflite_file.name)
    
    # Salva metadados
    metadata = {
        'classes': label_encoder.classes_.tolist(),
        'input_shape': input_shape,
        'test_accuracy': float(test_acc),
        'test_loss': float(test_loss)
    }
    with open(DRIVE_MODEL_DIR / 'model_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"✅ Modelo salvo no Google Drive: {DRIVE_MODEL_DIR}")
except Exception as e:
    print(f"⚠️ Erro ao salvar no Google Drive: {e}")

# Opção 2: Preparar download direto
print("\n📥 Para fazer download do modelo:")
print("  1. Clique com botão direito nos arquivos abaixo")
print("  2. Selecione 'Download'")
print(f"\nArquivos disponíveis em: {TFLITE_DIR}")

# Lista arquivos para download
for file in TFLITE_DIR.glob('*.tflite'):
    print(f"  - {file.name}")

# Cria arquivo ZIP com modelo e metadados
ZIP_PATH = MODEL_DIR / 'cough_classifier_model.zip'
with zipfile.ZipFile(ZIP_PATH, 'w') as zipf:
    for file in TFLITE_DIR.glob('*.tflite'):
        zipf.write(file, file.name)
    if (OUTPUT_DIR / 'metadata.json').exists():
        zipf.write(OUTPUT_DIR / 'metadata.json', 'metadata.json')

print(f"\n✅ Arquivo ZIP criado: {ZIP_PATH}")
print(f"   Tamanho: {os.path.getsize(ZIP_PATH) / (1024 * 1024):.2f} MB")

print("\n🎉 Processo completo concluído!")
print("   O modelo está pronto para ser usado em aplicações móveis!")
